# 🧠 Support Ticket Classification Model  
### Category & Priority Prediction using Local AI-Labeled Data

---

## 📌 Purpose of This Notebook

This notebook trains and evaluates a **machine learning–based support ticket classifier** using the **latest locally categorized and prioritized dataset**.

The model is designed to:
- Predict the **ticket category** (e.g., infrastructure, billing, security, hardware)
- Assign a **business-aligned priority** (High / Medium / Low)
- Handle real-world edge cases such as:


In [13]:
import torch

import pandas as pd 
import sklearn 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import joblib
import os



torch.__version__

'2.10.0+cu126'

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Using pandas, Let's read the data

In [15]:
data = pd.read_csv('classified_tickets.csv')
data.head()

,id,description,category,priority
0,1,"Account Disruption Dear Customer Support Team,...",account,High
1,2,Query About Smart Home System Integration Feat...,other,Low
2,3,Inquiry Regarding Invoice Details Dear Custome...,billing,Low
3,4,Question About Marketing Agency Software Compa...,other,Low
4,5,"Feature Query Dear Customer Support,\n\nI hope...",other,Low


### Splitting the data to train-test split

In [16]:
train_df , test_df = train_test_split(data,
                                    test_size=0.2,
                                    random_state=42)
train_df.head()

,id,description,category,priority
12909,26539,Support Request for Safeguarding Medical Data ...,security,Low
8373,16526,Business Growth Assistance Looking to enhance ...,other,Low
23108,58735,Hospital Data Security Incident Report Custome...,security,High
4678,9438,Request for Updated Billing Information We are...,billing,Low
20810,54362,Problem with Signal Integration Dear Customer ...,integration,Low


### Initializing TF-IDF Vectorizer

In [17]:
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000
)

### Initializing Priority and category encoder

In [18]:
priority_encoder = LabelEncoder()

y_train_priority = priority_encoder.fit_transform(train_df["priority"])
y_test_priority = priority_encoder.transform(test_df["priority"])


In [19]:
category_encoder = LabelEncoder()

y_train_category = category_encoder.fit_transform(train_df["category"])
y_test_category = category_encoder.transform(test_df["category"])
y_train_category[:10], y_test_category[:10]

(array([9, 8, 9, 1, 6, 9, 8, 8, 9, 8]), array([5, 9, 9, 9, 6, 8, 8, 6, 9, 8]))

In [20]:
X_train = tfidf.fit_transform(train_df["description"])
X_test = tfidf.transform(test_df["description"])
X_train.shape, X_test.shape, y_train_priority.shape, y_test_priority.shape, y_train_category.shape, y_test_category.shape


((19696, 20000), (4925, 20000), (19696,), (4925,), (19696,), (4925,))

In [21]:
priority_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"  # VERY important for priority
)
priority_model.fit(X_train, y_train_priority)
priority_preds = priority_model.predict(X_test)
print(classification_report(y_test_priority, priority_preds))

              precision    recall  f1-score   support

           0       0.89      0.86      0.88      1475
           1       0.96      0.89      0.92      2187
           2       0.78      0.90      0.84      1263

    accuracy                           0.89      4925
   macro avg       0.88      0.89      0.88      4925
weighted avg       0.89      0.89      0.89      4925



In [22]:
base_svc = LinearSVC()

category_model = CalibratedClassifierCV(
    base_svc,
    method="sigmoid",
    cv=5               
)
category_model.fit(X_train, y_train_category)
category_preds = category_model.predict(X_test)
print(classification_report(y_test_category, category_preds))


              precision    recall  f1-score   support

           0       0.72      0.50      0.59        72
           1       0.94      0.93      0.94       233
           2       0.78      0.80      0.79       299
           3       0.86      0.75      0.80         8
           4       0.89      0.57      0.69        30
           5       0.85      0.91      0.88       559
           6       0.88      0.89      0.88      1196
           7       0.88      0.76      0.82       100
           8       0.91      0.90      0.91      1297
           9       0.97      0.96      0.96      1131

    accuracy                           0.90      4925
   macro avg       0.87      0.80      0.83      4925
weighted avg       0.90      0.90      0.90      4925



In [23]:
import random
import numpy as np

def test_single_ticket(model, vectorizer, text, label_enc_cat, label_enc_pri):
    # Vectorize
    X = vectorizer.transform([text])

    # Predict
    cat_pred = model['category'].predict(X)[0]
    pri_pred = model['priority'].predict(X)[0]

    # Decode
    category = label_enc_cat.inverse_transform([cat_pred])[0]
    priority = label_enc_pri.inverse_transform([pri_pred])[0]

    return category, priority


In [24]:
text = """
Office applications fail to open after macOS update.
"""

category, priority = test_single_ticket(
    {
        'category': category_model,
        'priority': priority_model
    },
    tfidf,
    text,
    category_encoder,
    priority_encoder
)
ACCESS_BLOCK_KEYWORDS = {
    "fail to open",
    "cannot open",
    "unable to open",
    "won't open",
    "does not open",
    "blocked",
    "cannot access"
}
if any(k in text for k in ACCESS_BLOCK_KEYWORDS):
    if priority == "Low":
        priority = "Medium"

print("CATEGORY:", category)
print("PRIORITY:", priority)


CATEGORY: integration
PRIORITY: Medium


In [25]:
priority_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])




In [26]:
category_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("model", CalibratedClassifierCV(
        LinearSVC(),
        method="sigmoid",
        cv=5
    ))
])


In [27]:
priority_pipeline.fit(train_df["description"], y_train_priority)

category_pipeline.fit(train_df["description"], y_train_category)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [28]:
# Priority
priority_preds = priority_pipeline.predict(test_df["description"])
print("Priority Report:")
print(classification_report(y_test_priority, priority_preds))

# Category
category_preds = category_pipeline.predict(test_df["description"])
print("Category Report:")
print(classification_report(y_test_category, category_preds))


Priority Report:
              precision    recall  f1-score   support

           0       0.89      0.86      0.88      1475
           1       0.96      0.89      0.92      2187
           2       0.78      0.90      0.84      1263

    accuracy                           0.89      4925
   macro avg       0.88      0.89      0.88      4925
weighted avg       0.89      0.89      0.89      4925

Category Report:
              precision    recall  f1-score   support

           0       0.72      0.50      0.59        72
           1       0.94      0.93      0.94       233
           2       0.78      0.80      0.79       299
           3       0.86      0.75      0.80         8
           4       0.89      0.57      0.69        30
           5       0.85      0.91      0.88       559
           6       0.88      0.89      0.88      1196
           7       0.88      0.76      0.82       100
           8       0.91      0.90      0.91      1297
           9       0.97      0.96      0.96 

In [29]:
print(priority_pipeline.predict_proba(["server is down"]))
print(category_pipeline.predict_proba(["cannot login to account"]))


[[0.45679428 0.0087403  0.53446542]]
[[5.86489356e-01 1.18870074e-02 1.02976910e-04 2.24036778e-04
  5.71260179e-04 1.92095709e-07 1.34797549e-04 9.28042493e-04
  3.99238026e-01 4.24304893e-04]]


In [30]:
os.makedirs("backend/model", exist_ok=True)

joblib.dump(priority_pipeline, "backend/model/priority_pipeline.pkl")
joblib.dump(priority_encoder, "backend/model/priority_encoder.pkl")

joblib.dump(category_pipeline, "backend/model/category_pipeline.pkl")
joblib.dump(category_encoder, "backend/model/category_encoder.pkl")


['backend/model/category_encoder.pkl']